# LangChain Core: `invoke`, `stream`, `batch`

Every LangChain chat model is a **Runnable**. That gives every model the same 3 core methods:

| Method | What it does |
|---|---|
| `invoke()` | Send input, wait for the full response |
| `stream()` | Send input, get the response chunk by chunk |
| `batch()` | Send multiple independent inputs at once (parallel) |

Each method accepts input in **multiple formats**: a plain string, a list of dicts, a list of message objects, or a list of tuples. This notebook walks through all of them using `langchain_openai.ChatOpenAI` with model `gpt-5.4-nano`.

## Setup

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-5.4-nano")

In [2]:
query1= "what is ai in 1 line" # human message or user message

query2="""
OpenAI & Hugging Face are popular AI platforms.
Visit: https://huggingface.co
Today is 28/07/2026 🚀
"""

In [3]:
ai_message= llm.invoke(query1)

In [4]:
ai_message

AIMessage(content='AI (Artificial Intelligence) is technology that enables machines to perform tasks that typically require human intelligence, like learning, reasoning, and recognizing patterns.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 13, 'total_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-E9SJreIFEFNyTOXVNEe8SBKUpw5iR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fd146-f46d-7381-96e6-400915f28460-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 31, 'total_tokens': 44, 'input_token_details': {'audio': 0, 'cache_rea

In [6]:
ai_message.content

'AI (Artificial Intelligence) is computer technology that enables machines to learn from data and perform tasks that typically require human intelligence.'

In [7]:
ai_message.response_metadata

{'token_usage': {'completion_tokens': 27,
  'prompt_tokens': 13,
  'total_tokens': 40,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0,
   'cache_write_tokens': None,
   'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-5.4-nano-2026-03-17',
 'system_fingerprint': None,
 'id': 'chatcmpl-E6dtLl5nX2YPvQRGOVTQA9zrYH5WU',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

## 1. `invoke()`

Returns a single `AIMessage` after the model finishes generating.

### Format 1 — plain string

Simplest input. Treated as a single human message.

In [5]:
response = llm.invoke("Why is the sky blue?")
print(response.content)

The sky looks blue mainly because of **Rayleigh scattering**.

- Sunlight contains many colors (a mix of wavelengths).
- When it enters Earth’s atmosphere, molecules in the air (like nitrogen and oxygen) scatter the light.
- **Shorter wavelengths** (blue/violet) scatter much more strongly than longer wavelengths (red).
- We therefore receive lots of scattered **blue light** from all directions across the sky.

Why not violet? Although violet scatters even more, **our eyes are less sensitive to violet** and some violet light is also absorbed/scattered differently in the atmosphere, so **blue dominates** what we perceive.


In [11]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

template1 = [
    SystemMessage("You are a poem expert, answer poem coding related querieselse say sorry i dont know"),
    HumanMessage("Write a poem about spring"),
]


In [12]:
template1 = [
    SystemMessage("You are a poem expert, answer poem coding related querieselse say sorry i dont know"),
    HumanMessage("Write a poem about spring"),
]


In [13]:
response = llm.invoke(template1)

In [14]:
print(response.content)

Spring arrives in borrowed light,  
A shy green stitch across the sky—  
The world exhales, and buds take flight  
From winters’ closed-up, brittle sigh.

Rivers learn new laughter, clear,  
Threading through stones worn smooth with time;  
The sidewalks brighten, soft with cheer,  
As puddled suns begin to climb.

Birdsong stitches morning into day,  
Quick as a thought that won’t stay still;  
Even roots, beneath the thawing clay,  
Unfold their hope within their will.

And in the garden’s wake of rain,  
Where every seed believes in more,  
The air tastes sweet—like something gained—  
As if the earth is saying: *Come for.*


In [15]:
dict_template= [
    {"role": "system", "content": "You are a poem expert, answer poem coding related querieselse say sorry i dont know"},
    {"role": "user", "content": "Write a poem about spring"},
]
response = llm.invoke(dict_template)

In [16]:
system_msg = SystemMessage("You are a helpful coding assistant.")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = llm.invoke(messages)

In [17]:
print(response.content)

Creating a REST API usually boils down to: choose a stack, define your endpoints/resources, implement CRUD (where applicable), handle request/response formats (typically JSON), add validation/auth, and document/test it.

Below is a practical, step-by-step guide you can follow with any language/framework.

---

## 1) Decide the scope and resources
Think in terms of **resources** (nouns), not actions.

Example resources:
- `users`
- `products`
- `orders`

Example endpoints (typical REST):
- `GET /products` → list
- `POST /products` → create
- `GET /products/{id}` → fetch one
- `PUT/PATCH /products/{id}` → update
- `DELETE /products/{id}` → delete

---

## 2) Choose a framework / stack
Common choices:
- **Node.js**: Express, NestJS
- **Python**: Flask, FastAPI (very popular for APIs)
- **Java**: Spring Boot
- **Go**: net/http, Gin
- **.NET**: ASP.NET Core

If you’re not sure, **FastAPI** (Python) or **Express** (Node) are great starting points.

---

## 3) Define your API contract
Pick:
-

### Format 2 — dictionary (OpenAI chat format)

A list of `{"role": ..., "content": ...}` dicts. Roles: `system`, `user`, `assistant`.

In [18]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to hindi."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "mujhe programming acchi lagti hai"},
    {"role": "user", "content": "Translate: I love building applications."},
]

response = llm.invoke(conversation)
print(response.content)

मुझे एप्लिकेशन बनाना अच्छा लगता है।


In [19]:
conversation2 = [
    SystemMessage("You are a helpful assistant that translates English to hindi in roman english only."),
    HumanMessage("Translate: I love programming."),
    AIMessage("mujhe programming acchi lagti hai"),
    HumanMessage("Translate: I love building applications.")
]


In [20]:

response = llm.invoke(conversation2)
print(response.content)

mujhe applications banana bahut pasand hai


In [21]:
target_language="telugu"

conversation2 = [
    SystemMessage(f"You are a helpful assistant that translates English to {target_language} in roman english only."),
    HumanMessage("Translate: I love programming."),
]


response = llm.invoke(conversation2)
print(response.content)

Naku programming ante chala istam.


### Format 3 — message objects (`langchain_core.messages`)

Same conversation, using typed message classes instead of dicts.

In [22]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications."),
]

response = llm.invoke(conversation)
print(response.content)

J’adore créer des applications.


### Format 4 — tuples `(role, content)`

A shorthand: each message is a `(role, content)` tuple instead of a dict or object.

In [23]:
conversation = [
    ("system", "You are a helpful assistant that translates English to French."),
    ("human", "Translate: I love programming."),
]

response = llm.invoke(conversation)
print(response.content)

J’aime programmer.


## 2. `stream()`

Returns an iterator of `AIMessageChunk` pieces instead of one final message. Same input formats as `invoke()` work here too.

### String input, streamed

In [24]:
for chunk in llm.stream("Why do parrots have colorful feathers?"):
    print(chunk.content, end="", flush=True)

Parrots have colorful feathers mainly for two reasons: **communication** and **survival**.

- **Communication (social signals):** Bright colors help parrots recognize each other, attract mates, and signal health or status. Many species use plumage to show fitness during courtship.
- **Camouflage and protection:** Some coloration helps parrots blend into their surroundings or blend into specific light conditions in the forest canopy.
- **Reflecting light for display:** A lot of parrot “color” is produced not just by pigment but also by feather micro-structure that reflects light in vivid ways—especially for blues and greens.
- **Different colors can mean different things:** In some species, brighter or more uniform coloration can indicate better nutrition, stronger immune function, or overall better condition.

So, while the exact mix varies by species, the big idea is that **colorful feathers are useful for getting mates and staying safe** (and they often work through both pigments and

## 3. `batch()`

Sends multiple **independent** inputs in parallel. Input is a list where each item can itself be any of the formats above (string, dict list, message list, tuple list).

### List of strings

In [25]:
responses = llm.batch([
    "What is the capital of India?",
    "What is the cultural food of saudi?",
    "What is my name?",
])

for r in responses:
    print(r.content)

The capital of India is **New Delhi**.
Saudi Arabia’s “cultural” foods are commonly based on regional Arab cuisine, Bedouin/heritage dishes, and ingredients like dates, rice, lamb/chicken, and spices. Some of the best-known traditional foods are:

- **Kabsa (كبسة)**: A fragrant spiced **rice** dish (often with chicken or lamb) cooked with tomatoes, garlic, onions, and a blend of spices.
- **Mandi (مندي)**: Similar to kabsa but **more intensely flavored**, traditionally cooked underground or in special ovens; typically with meat and rice.
- **Jareesh (جريش)**: **Cracked wheat** cooked into a thick porridge, often topped with meat (common especially in the central/eastern regions).
- **Harees/Hareesa (هريس)**: A slow-cooked wheat-and-meat dish (often lamb or chicken), usually served during special occasions.
- **Al Madfoon (المدفون)**: Meat and vegetables slow-cooked (often buried or covered) with spices for a deep flavor.
- **Hummus and mutabal (مقبلات)**: Very common starters across th

### List of conversations (dict format), with `config`

`config={"max_concurrency": N}` limits how many calls run in parallel.

In [36]:
inputs = [
    [{"role": "user", "content": "Summarize photosynthesis in one line."}],
    [{"role": "user", "content": "Summarize gravity in one line."}],
    [{"role": "user", "content": "What is the cultural food of saudi?"}]
]

responses = llm.batch(inputs, config={"max_concurrency": 100})

for r in responses:
    print(r.content)

Photosynthesis is the process by which plants convert sunlight into chemical energy (glucose) by using water and carbon dioxide, releasing oxygen.
Gravity is the attractive force between masses that pulls objects toward each other.
Saudi Arabia doesn’t have just one single “cultural food,” but a few foods are widely considered traditional and commonly associated with Saudi cuisine:

- **Mandi** (or **Maqli/Mandi rice**): Roasted meat (usually lamb, chicken, or camel) cooked with **spiced rice**—often served on Fridays and for celebrations.  
- **Kabsa**: Similar to mandi but typically **chicken or lamb with spiced rice** and aromatic seasonings (saffron/black lime/cardamom/spices depending on region).  
- **Kofta**: Grilled or pan-cooked spiced meat patties or meatballs, often served with bread and sauces.  
- **Harees** (or **Harisa**): A hearty wheat-and-meat dish, slow-cooked until smooth and thick—popular during special occasions like Ramadan and holidays.  
- **Hummus, Mutabbal, a